# 05 — Pipeline ALPR, Evaluasi Akhir, Paket Deploy
Menggabungkan YOLOv8n (OpenVINO) + OCR fine-tune (ONNX via OpenVINO) + validasi pola plat Indonesia menjadi
modul `alpr_engine.py` yang **sama persis** dengan yang nanti dipakai di mini PC.

Modul ini **tidak membutuhkan PyTorch maupun PaddlePaddle**: cukup `openvino`, `opencv-python`, `numpy`, `pyyaml`.

**Runtime:** CPU saja (sesi baru). Latensi di Colab CPU hanya gambaran; angka final harus diukur ulang di Beelink T4 Pro.

## 1. Drive dan instalasi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/0-KEPSTON/model_alpr'
OCR = f'{ROOT}/data/ocr'
!pip install -q -U openvino opencv-python-headless pyyaml

In [ ]:
from pathlib import Path
import sys
CODE = Path(f'{ROOT}/code'); CODE.mkdir(parents=True, exist_ok=True)
(CODE / 'plate_utils.py').write_text('"""Utilitas plat nomor Indonesia: pembersihan, koreksi karakter per posisi, validasi pola."""\nimport re\n\nLETTER_FIX = {\'0\': \'O\', \'1\': \'I\', \'2\': \'Z\', \'4\': \'A\', \'5\': \'S\', \'6\': \'G\', \'7\': \'T\', \'8\': \'B\'}\nDIGIT_FIX = {\'O\': \'0\', \'D\': \'0\', \'Q\': \'0\', \'U\': \'0\', \'I\': \'1\', \'L\': \'1\', \'J\': \'1\',\n             \'Z\': \'2\', \'A\': \'4\', \'S\': \'5\', \'G\': \'6\', \'T\': \'7\', \'B\': \'8\'}\nPATTERN = re.compile(r\'^([A-Z]{1,2})([0-9]{1,4})([A-Z]{0,3})$\')\n\n# Kode wilayah yang umum dipakai. Dipakai sebagai penalti lunak (bukan penolakan),\n# sehingga kode yang tidak ada di daftar tetap bisa diterima bila tidak ada pilihan lain.\nREGION_CODES = set(\'A B D E F G H K L M N P R S T W Z\'.split()) | set(\n    \'AA AB AD AE AG BA BB BD BE BG BH BK BL BM BN BP DA DB DC DD DE DG DH DK DL DM DN DR DS DT \'\n    \'EA EB ED KB KH KT KU PA PB\'.split())\n\n\ndef clean(text):\n    """Huruf kapital, buang semua selain A-Z dan 0-9."""\n    return re.sub(r\'[^A-Z0-9]\', \'\', str(text).upper())\n\n\ndef fix_plate(text):\n    """Cari pembagian [huruf 1-2][angka 1-4][huruf 0-3] dengan koreksi karakter paling sedikit.\n    Mengembalikan string tanpa spasi (mis. \'B1234XYZ\') atau None bila tidak bisa dibentuk."""\n    s = clean(text)\n    if len(s) > 9 and s[-4:].isdigit():        # sisa baris masa berlaku (MM.YY) ikut terbaca\n        s = s[:-4]\n    best = None\n    for a in (1, 2):\n        for b in range(1, 5):\n            c = len(s) - a - b\n            if c < 0 or c > 3:\n                continue\n            out, cost, ok = [], 0, True\n            for i, ch in enumerate(s):\n                if i < a or i >= a + b:          # posisi huruf\n                    if ch.isalpha():\n                        out.append(ch)\n                    elif ch in LETTER_FIX:\n                        out.append(LETTER_FIX[ch]); cost += 1\n                    else:\n                        ok = False; break\n                else:                             # posisi angka\n                    if ch.isdigit():\n                        out.append(ch)\n                    elif ch in DIGIT_FIX:\n                        out.append(DIGIT_FIX[ch]); cost += 1\n                    else:\n                        ok = False; break\n            if ok:\n                if \'\'.join(out[:a]) not in REGION_CODES:\n                    cost += 2                     # kode wilayah tidak dikenal\n                key = (cost, -b)\n                if best is None or key < best[0]:\n                    best = (key, \'\'.join(out))\n    return best[1] if best else None\n\n\ndef is_valid(plate):\n    return bool(plate) and bool(PATTERN.match(plate))\n\n\ndef format_plate(plate):\n    """\'B1234XYZ\' -> \'B 1234 XYZ\'"""\n    m = PATTERN.match(plate or \'\')\n    return \' \'.join(g for g in m.groups() if g) if m else (plate or \'\')\n\n\ndef levenshtein(a, b):\n    a, b = a or \'\', b or \'\'\n    prev = list(range(len(b) + 1))\n    for i, ca in enumerate(a, 1):\n        cur = [i]\n        for j, cb in enumerate(b, 1):\n            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))\n        prev = cur\n    return prev[-1]\n')
(CODE / 'alpr_engine.py').write_text('"""LocalALPR: deteksi plat (YOLOv8n, OpenVINO) + OCR (PP-OCRv5 fine-tune, ONNX via OpenVINO).\nTidak membutuhkan PyTorch maupun PaddlePaddle saat dijalankan, sehingga bisa dipakai di mini PC\ntanpa GPU dan tanpa dukungan AVX. Kebutuhan: openvino, opencv-python, numpy, pyyaml."""\nimport time\nfrom collections import Counter\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport openvino as ov\nimport yaml\n\nfrom plate_utils import fix_plate, is_valid, format_plate\n\nCROP_MARGIN_X, CROP_MARGIN_Y = 0.04, 0.10     # harus sama dengan notebook 03\n\n\ndef crop_plate(img, x1, y1, x2, y2):\n    h, w = img.shape[:2]\n    bw, bh = x2 - x1, y2 - y1\n    x1 = int(max(0, x1 - bw * CROP_MARGIN_X)); x2 = int(min(w, x2 + bw * CROP_MARGIN_X))\n    y1 = int(max(0, y1 - bh * CROP_MARGIN_Y)); y2 = int(min(h, y2 + bh * CROP_MARGIN_Y))\n    return img[y1:y2, x1:x2]\n\n\ndef _load_charset(ocr_dir):\n    """Urutan karakter keluaran OCR: index 0 = blank (CTC), lalu kamus, lalu spasi bila dipakai."""\n    ocr_dir = Path(ocr_dir)\n    chars = None\n    yml = ocr_dir / \'inference.yml\'\n    if yml.exists():\n        cfg = yaml.safe_load(yml.read_text(encoding=\'utf-8\')) or {}\n        pp = cfg.get(\'PostProcess\', {}) or {}\n        chars = pp.get(\'character_dict\')\n    if not chars:\n        chars = (ocr_dir / \'ppocrv5_dict.txt\').read_text(encoding=\'utf-8\').splitlines()\n    return [\'blank\'] + [c for c in chars]\n\n\nclass LocalALPR:\n    def __init__(self, yolo_xml, ocr_onnx, ocr_dir, imgsz=640, conf=0.5, iou=0.5, device=\'CPU\'):\n        core = ov.Core()\n        self.det = core.compile_model(str(yolo_xml), device)\n        self.ocr = core.compile_model(str(ocr_onnx), device)\n        self.imgsz, self.conf, self.iou = imgsz, conf, iou\n        self.chars = _load_charset(ocr_dir)\n\n    # ---------- deteksi ----------\n    def _letterbox(self, img):\n        s = self.imgsz\n        h, w = img.shape[:2]\n        r = min(s / h, s / w)\n        nh, nw = round(h * r), round(w * r)\n        canvas = np.full((s, s, 3), 114, dtype=np.uint8)\n        top, left = (s - nh) // 2, (s - nw) // 2\n        canvas[top:top + nh, left:left + nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)\n        return canvas, r, left, top\n\n    def detect(self, frame):\n        canvas, r, left, top = self._letterbox(frame)\n        x = canvas[:, :, ::-1].transpose(2, 0, 1)[None].astype(np.float32) / 255.0\n        pred = self.det([x])[self.det.output(0)][0].T          # (N, 5): cx, cy, w, h, score\n        pred = pred[pred[:, 4] >= self.conf]\n        if len(pred) == 0:\n            return []\n        H, W = frame.shape[:2]\n        boxes, scores = [], []\n        for cx, cy, bw, bh, sc in pred:\n            x1 = (cx - bw / 2 - left) / r; y1 = (cy - bh / 2 - top) / r\n            boxes.append([float(x1), float(y1), float(bw / r), float(bh / r)]); scores.append(float(sc))\n        keep = cv2.dnn.NMSBoxes(boxes, scores, self.conf, self.iou)\n        out = []\n        for i in np.array(keep).flatten():\n            x1, y1, bw, bh = boxes[i]\n            out.append((max(0, x1), max(0, y1), min(W, x1 + bw), min(H, y1 + bh), scores[i]))\n        return sorted(out, key=lambda b: -b[4])\n\n    # ---------- OCR ----------\n    def read_text(self, crop):\n        h, w = crop.shape[:2]\n        if h == 0 or w == 0:\n            return \'\', 0.0\n        H, MAXW = 48, 320\n        nw = min(MAXW, max(1, int(np.ceil(H * w / h))))\n        img = cv2.resize(crop, (nw, H)).astype(np.float32)\n        img = (img / 255.0 - 0.5) / 0.5\n        pad = np.zeros((H, MAXW, 3), dtype=np.float32)\n        pad[:, :nw] = img\n        x = pad.transpose(2, 0, 1)[None]                       # BGR, CHW\n        prob = self.ocr([x])[self.ocr.output(0)][0]             # (T, C)\n        idx, mx = prob.argmax(-1), prob.max(-1)\n        text, confs, prev = [], [], -1\n        for t, k in enumerate(idx):\n            if k != prev and k != 0 and k < len(self.chars):\n                text.append(self.chars[k]); confs.append(float(mx[t]))\n            prev = k\n        return \'\'.join(text), (float(np.mean(confs)) if confs else 0.0)\n\n    # ---------- pipeline ----------\n    def recognize(self, frame):\n        t0 = time.perf_counter()\n        results = []\n        try:\n            for x1, y1, x2, y2, ds in self.detect(frame):\n                raw, oc = self.read_text(crop_plate(frame, x1, y1, x2, y2))\n                plate = fix_plate(raw)\n                results.append({\n                    \'plate_text\': plate or \'\', \'plate_display\': format_plate(plate),\n                    \'valid\': is_valid(plate), \'raw_text\': raw,\n                    \'confidence\': round(oc * ds, 4), \'ocr_conf\': round(oc, 4), \'det_conf\': round(ds, 4),\n                    \'box\': [int(x1), int(y1), int(x2), int(y2)],\n                })\n            results.sort(key=lambda r: (not r[\'valid\'], -r[\'confidence\']))\n            return {\'success\': True, \'results\': results,\n                    \'latency_ms\': round((time.perf_counter() - t0) * 1000, 1)}\n        except Exception as e:\n            return {\'success\': False, \'error\': str(e), \'results\': []}\n\n    def recognize_vote(self, frames):\n        """Voting beberapa frame: ambil plat valid yang paling sering muncul."""\n        votes, conf = Counter(), {}\n        for f in frames:\n            res = self.recognize(f)\n            if res[\'success\'] and res[\'results\'] and res[\'results\'][0][\'valid\']:\n                p = res[\'results\'][0][\'plate_text\']\n                votes[p] += 1\n                conf.setdefault(p, []).append(res[\'results\'][0][\'confidence\'])\n        if not votes:\n            return None\n        plate, n = votes.most_common(1)[0]\n        return {\'plate_text\': plate, \'plate_display\': format_plate(plate),\n                \'votes\': n, \'frames\': len(frames), \'confidence\': float(np.mean(conf[plate]))}\n')
sys.path.insert(0, str(CODE))
import importlib, plate_utils; importlib.reload(plate_utils)
from plate_utils import clean, fix_plate, is_valid, format_plate, levenshtein
print(fix_plate('b 1234 xyz'), fix_plate('B1Z34XY2'), fix_plate('BI234XYZ 05.28'))

## 2. Muat model

In [ ]:
import glob, importlib
import alpr_engine; importlib.reload(alpr_engine)
from alpr_engine import LocalALPR, crop_plate

OCR_DIR = f'{ROOT}/models/ppocrv5_plate_onnx'
OCR_ONNX = sorted(glob.glob(f'{OCR_DIR}/*.onnx'))[0]
def yolo_xml(sz):
    return sorted(glob.glob(f'{ROOT}/models/yolov8n_plate_openvino_{sz}/*.xml'))[0]

alpr = LocalALPR(yolo_xml(640), OCR_ONNX, OCR_DIR, imgsz=640)
print('Model siap | jumlah karakter OCR:', len(alpr.chars))

## 3. Cek OCR ONNX sama dengan model Paddle
Membandingkan hasil baca ONNX dengan label pada 50 potongan test. Kalau exact match jauh lebih rendah dari hasil notebook 04,
berarti ada perbedaan pra-proses atau urutan karakter yang harus diperiksa sebelum lanjut.

In [ ]:
import cv2, pandas as pd
items = [ln.rstrip('\n').split('\t') for ln in open(f'{OCR}/test_list.txt') if ln.strip()]
ok = 0
for rel, gt in items[:50]:
    raw, _ = alpr.read_text(cv2.imread(f'{OCR}/{rel}'))
    ok += (fix_plate(raw) == gt)
print(f'OCR ONNX pada 50 potongan test: exact match = {ok / min(50, len(items)):.1%}')

## 4. Evaluasi end-to-end (gambar utuh → teks plat)
Untuk setiap gambar test, pipeline lengkap dijalankan (deteksi → potong → OCR → `fix_plate`).
Sebuah plat dihitung **benar** kalau teks labelnya ada di hasil pipeline untuk gambar itu.
Ini angka utama untuk laporan (target **≥ 90%**).

In [ ]:
import numpy as np, time
from pathlib import Path

lab = pd.read_csv(f'{OCR}/labels.csv', dtype=str).fillna('')
lab = lab[lab['status'] == 'ok']

def end_to_end(split, base_dir, sz):
    eng = LocalALPR(yolo_xml(sz), OCR_ONNX, OCR_DIR, imgsz=sz)
    part = lab[lab['split'] == split]
    rows, lat = [], []
    for src, g in part.groupby('source'):
        path = (Path(base_dir) / src) if split != 'test_lokal' else (Path(f'{ROOT}/data/raw') / src)
        img = cv2.imread(str(path))
        if img is None:
            continue
        res = eng.recognize(img); lat.append(res.get('latency_ms', np.nan))
        preds = {r['plate_text'] for r in res['results']}
        for gt in g['text']:
            rows.append({'source': src, 'gt': gt, 'pred': ','.join(sorted(preds)), 'benar': gt in preds})
    r = pd.DataFrame(rows)
    if r.empty:
        print(split, ': tidak ada data'); return None
    print(f'[{split} | imgsz {sz}] akurasi plat = {r.benar.mean():.1%} (n = {len(r)}) | '
          f'latensi rata-rata = {np.nanmean(lat):.0f} ms | p95 = {np.nanpercentile(lat, 95):.0f} ms')
    r.to_csv(f'{ROOT}/eval/e2e_{split}_{sz}.csv', index=False)
    return r

results = {}
for sz in [640, 416]:
    for split in ['test', 'test_lokal']:
        results[(split, sz)] = end_to_end(split, f'{ROOT}/data/detect', sz)

### Contoh hasil pipeline

In [ ]:
import random
import matplotlib.pyplot as plt
srcs = random.sample(sorted(lab[lab['split'] == 'test']['source'].unique()), 8)
plt.figure(figsize=(20, 10))
for k, src in enumerate(srcs):
    img = cv2.imread(f'{ROOT}/data/detect/{src}')
    res = alpr.recognize(img)
    for r in res['results']:
        x1, y1, x2, y2 = r['box']
        col = (0, 200, 0) if r['valid'] else (0, 0, 255)
        cv2.rectangle(img, (x1, y1), (x2, y2), col, 2)
        cv2.putText(img, r['plate_display'] or r['raw_text'], (x1, max(20, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, col, 2)
    plt.subplot(2, 4, k + 1); plt.imshow(img[:, :, ::-1]); plt.axis('off')
    plt.title(f"{res.get('latency_ms', '-')} ms", fontsize=10)
plt.tight_layout(); plt.show()

## 5. Paket deploy untuk mini PC
Semua yang dibutuhkan mini PC disalin ke `models/deploy/`. Salin folder ini ke mini PC (misalnya ke folder program Smart Gate).

Di mini PC (Ubuntu): `pip install openvino opencv-python numpy pyyaml`, lalu:
```python
import sys; sys.path.insert(0, 'deploy')
from alpr_engine import LocalALPR
alpr = LocalALPR('deploy/yolo/best.xml', 'deploy/ocr/inference.onnx', 'deploy/ocr', imgsz=416)
hasil = alpr.recognize(frame)      # frame BGR dari OpenCV
```
Pilih `imgsz` (640 atau 416) berdasarkan hasil sel 4: ambil yang akurasinya masih ≥ target dengan latensi terkecil.

In [ ]:
import shutil, os
DEP = Path(f'{ROOT}/models/deploy')
if DEP.exists():
    shutil.rmtree(DEP)
DEP.mkdir(parents=True)
shutil.copytree(f'{ROOT}/models/yolov8n_plate_openvino_640', DEP / 'yolo_640')
shutil.copytree(f'{ROOT}/models/yolov8n_plate_openvino_416', DEP / 'yolo_416')
shutil.copytree(OCR_DIR, DEP / 'ocr')
for f in ['alpr_engine.py', 'plate_utils.py']:
    shutil.copy(f'{ROOT}/code/{f}', DEP)
for root, _, files in os.walk(DEP):
    for f in files:
        print(os.path.relpath(os.path.join(root, f), DEP))

---
### Checklist 05
- [ ] OCR ONNX pada potongan test setara dengan hasil fine-tune di notebook 04
- [ ] Akurasi plat end-to-end pada test **≥ 90%** (catat juga test_lokal bila ada)
- [ ] Tentukan `imgsz` (640/416) dan catat latensinya
- [ ] Folder `models/deploy/` siap disalin ke mini PC
- [ ] **Ukur ulang latensi langsung di Beelink T4 Pro** sebelum menulis angka final di laporan

**Integrasi berikutnya:** ganti pemanggilan `PlateRecognizerAPI` di `main.py` dengan `LocalALPR`
(struktur hasil `results[0]['plate_text']` dan `['confidence']` sudah disamakan).